This is part 13 of a tutorial series. We recommend following them in order, starting with [Part 0: Welcome to `musica`](0.%20Welcome%20to%20MUSICA.ipynb). We also recommend reading [Part 12: Introduction to DAE Solving](12.%20dae_solving_intro.ipynb) first for the conceptual background.

# MIAM Processes, Constraints, and Representations

MIAM (Model-Independent Aerosol Module) provides the building blocks for coupled gas-aerosol-cloud chemistry in `musica`. In this tutorial, we'll walk through every MIAM component — **representations**, **processes**, and **constraints** — with working code examples. We'll build up a system incrementally, adding one constraint at a time, which is the recommended approach for developing and debugging MIAM configurations.

## 1. Importing Libraries

In [1]:
import math
import musica
import musica.mechanism_configuration as mc
from musica.micm import MICM, SolverType, SolverState
from musica.miam import (
    # Constants
    HenrysLawConstant,
    EquilibriumConstant,
    ArrheniusRateConstant,
    # Representations
    UniformSection,
    SingleMomentMode,
    TwoMomentMode,
    # Processes
    DissolvedReaction,
    DissolvedReversibleReaction,
    HenryLawPhaseTransfer,
    # Constraints
    HenryLawEquilibriumConstraint,
    DissolvedEquilibriumConstraint,
    LinearConstraint,
    LinearConstraintTerm,
    # Model
    Model,
)
import matplotlib.pyplot as plt
import numpy as np

## 2. Architecture: How MIAM Integrates with MICM

In `musica`, gas-phase chemistry is defined through a `Mechanism` (species, phases, reactions). MIAM extends this by adding **condensed-phase** chemistry (aerosol, cloud) through a `Model` object. The two are coupled at solver creation time:

```python
# Gas-phase: defined via mechanism_configuration
mechanism = mc.Mechanism(species=[...], phases=[gas], reactions=[...])

# Condensed-phase: defined via miam.Model
model = Model(
    name="my_model",
    species=[...],            # All species (gas + condensed)
    condensed_phases=[...],   # Cloud/aerosol phases (aqueous, organic, etc.)
    representations=[...],    # Cloud/aerosol geometry
    processes=[...],          # Kinetic reactions in condensed phase
    constraints=[...],        # Algebraic (DAE) constraints
)

# Coupled solver
micm = MICM(
    mechanism=mechanism,
    solver_type=SolverType.rosenbrock_dae4_standard_order,
    external_models=[model],
)
```

The `external_models` parameter tells MICM to incorporate MIAM's processes and constraints into the solver. When constraints are present, you **must** use a DAE-capable solver type (one of the `rosenbrock_dae*` variants).

## 3. Species and Phases

Let's define the species we'll use throughout this tutorial. This is a simplified SO₂ cloud chemistry system.

In [2]:
# Physical constants
MW_H2O = 0.018      # kg/mol
RHO_H2O = 1000.0    # kg/m³
C_H2O_M = 55.556    # mol/L (molarity of pure water — used in rate constant conversions)
R_GAS = 8.314       # J/(mol·K)
T0 = 298.15         # K (reference temperature)
M_ATM_TO_MOL_M3_PA = 1000.0 / 101325.0  # unit conversion factor

# Cloud liquid water content
# In MIAM, the solvent concentration is "moles of liquid water in all
# cloud droplets per cubic meter of air" — NOT the molarity of pure water.
# A typical cloud has LWC ≈ 0.3 g/m³ of air.
LWC = 0.3e-3           # kg/m³ of air (= 0.3 g/m³)
C_H2O = LWC / MW_H2O   # mol/m³ of air ≈ 0.017
f_v = C_H2O * MW_H2O / RHO_H2O  # liquid volume fraction (m³ liquid / m³ air)
print(f"Cloud LWC = {LWC*1e3:.1f} g/m³  →  C_H2O = {C_H2O:.4f} mol/m³  →  f_v = {f_v:.2e}")

# Conditions
T_INIT = 280.0      # K
P_INIT = 70000.0    # Pa

# ── Gas-phase species ──
so2_g = mc.Species(name="SO2")
gas = mc.Phase(name="gas", species=[so2_g])
mechanism = mc.Mechanism(species=[so2_g], phases=[gas], reactions=[])

# ── Aqueous-phase species ──
h2o = mc.Species(name="H2O")
h2o.molecular_weight_kg_mol = MW_H2O
h2o.density_kg_m3 = RHO_H2O

so2_aq = mc.Species(name="SO2_aq")   # Dissolved SO2
hp = mc.Species(name="Hp")           # H+
ohm = mc.Species(name="OHm")         # OH-
hso3m = mc.Species(name="HSO3m")     # HSO3-

aq_phase = mc.Phase(
    name="AQUEOUS",
    species=[h2o, so2_aq, hp, ohm, hso3m],
)

all_species = [so2_g, so2_aq, hp, ohm, hso3m, h2o]

print("Species and phases defined.")

Cloud LWC = 0.3 g/m³  →  C_H2O = 0.0167 mol/m³  →  f_v = 3.00e-07
Species and phases defined.


## 4. Representations: Cloud and Aerosol Geometry

A **representation** describes the size distribution and geometry of the condensed-phase particles (cloud droplets, aerosol particles). MIAM uses the representation to compute surface-area-weighted transfer rates and volume-weighted concentrations.

Three representations are available:

### 4a. `UniformSection` — Uniform Size Distribution

A single size bin with uniform number concentration between `min_radius` and `max_radius`. Typical use: **cloud droplets** where the drop size distribution is simple.

In [3]:
cloud = UniformSection(
    name="CLOUD",
    phase_names=["AQUEOUS"],  # Which phases live in these droplets
    min_radius=1e-6,          # 1 μm (meters)
    max_radius=1e-5,          # 10 μm
)
print(f"Cloud droplets: {cloud.min_radius*1e6:.0f}–{cloud.max_radius*1e6:.0f} μm")

Cloud droplets: 1–10 μm


### 4b. `SingleMomentMode` — Log-Normal with Fixed Number

A log-normal distribution with fixed geometric standard deviation and number concentration. Typical use: **accumulation-mode aerosol** with prescribed size parameters.

In [4]:
accum_mode = SingleMomentMode(
    name="ACCUMULATION",
    phase_names=["AQUEOUS"],
    geometric_mean_radius=0.1e-6,      # 100 nm
    geometric_standard_deviation=1.8,   # typical for accumulation mode
)
print(f"Accumulation mode: r_g = {accum_mode.geometric_mean_radius*1e9:.0f} nm, σ_g = {accum_mode.geometric_standard_deviation}")

Accumulation mode: r_g = 100 nm, σ_g = 1.8


### 4c. `TwoMomentMode` — Prognostic Number and Mass

A log-normal with fixed geometric standard deviation, but the mean radius is diagnosed from the prognostic number and mass concentrations. Used in models where both number and mass evolve (e.g., MAM4 in CESM).

In [5]:
# Shown for reference — we'll use UniformSection for the rest of this tutorial
two_moment = TwoMomentMode(
    name="AITKEN",
    phase_names=["AQUEOUS"],
    geometric_standard_deviation=1.6,
)
print(f"Aitken mode: σ_g = {two_moment.geometric_standard_deviation}")

Aitken mode: σ_g = 1.6


### Setting representation parameters on the state

After creating a state from the solver, you need to call `model.set_default_parameters(state)` to initialize the representation parameters (radii, standard deviations). These are stored as user-defined rate parameters with names like `CLOUD.MIN_RADIUS`, `CLOUD.MAX_RADIUS`, etc.

## 5. Processes: Kinetic Reactions in the Condensed Phase

MIAM provides three process types for condensed-phase kinetics.

### 5a. `HenryLawPhaseTransfer` — Kinetic Gas-to-Aqueous Transfer

This models the **kinetic** (diffusion-limited) transfer of a gas into cloud droplets. The rate depends on the diffusion coefficient, accommodation coefficient, and droplet surface area.

Use this when the transfer timescale is comparable to or longer than your solver timestep.

In [6]:
# Example: kinetic SO2 dissolution (we won't use this in our constraint-based model)
so2_transfer = HenryLawPhaseTransfer(
    condensed_phase_name="AQUEOUS",
    gas_species_name="SO2",
    condensed_species_name="SO2_aq",
    solvent_name="H2O",
    henrys_law_constant=HenrysLawConstant(
        hlc_ref=1.23 * M_ATM_TO_MOL_M3_PA,  # mol/(m³·Pa)
        c=3120.0,                           # K (temperature dependence)
    ),
    diffusion_coefficient=1.3e-9,     # m²/s (SO2 in air)
    accommodation_coefficient=0.11,   # mass accommodation coefficient
)
print(f"SO2 kinetic transfer: D = {so2_transfer.diffusion_coefficient:.1e} m²/s, α = {so2_transfer.accommodation_coefficient}")

SO2 kinetic transfer: D = 1.3e-09 m²/s, α = 0.11


> **Kinetic vs. Equilibrium**: For cloud chemistry, the transfer timescale for cloud-sized droplets (1–10 μm) is typically microseconds — much faster than the solver timestep. In that case, use a `HenryLawEquilibriumConstraint` instead (see Section 6a). Use `HenryLawPhaseTransfer` when the droplets are large (rain) or the species has a very low accommodation coefficient.

### 5b. `DissolvedReaction` — Irreversible Aqueous Reaction

An irreversible bimolecular reaction in the aqueous phase of the form:

$$\text{rate} = \frac{k(T) \cdot [R_1] \cdot [R_2]}{[\text{solvent}]^{n-1}}$$

where $n$ is the number of reactants. The division by the solvent concentration converts from the solution-phase rate expression (where concentrations are in mol/L) to the MIAM convention (mol/m³).

In [7]:
# HSO3- + H2O2(aq) → SO4-- + H2O + H+
# Literature rate constant: 7.45e7 M⁻¹s⁻¹ at 298 K
# We multiply by C_H2O_M to account for the /[solvent] normalization
r1 = DissolvedReaction(
    phase_name="AQUEOUS",
    reactant_names=["HSO3m", "SO2_aq"],   # Using SO2_aq as a stand-in for H2O2
    product_names=["Hp"],                 # Simplified products
    solvent_name="H2O",
    rate_constant=ArrheniusRateConstant(
        a=C_H2O_M * 7.45e7,  # pre-exponential [mol/m³ units]
        c=4430.0,             # K (Ea/R)
    ),
)
print("DissolvedReaction created.")

DissolvedReaction created.


> **Rate constant units**: Literature aqueous-phase rate constants are typically in M⁻¹s⁻¹ (where M = mol/L). MIAM works in mol/m³. The `ArrheniusRateConstant` value should be multiplied by`C_H2O_M` (55.556 mol/L) for each concentration power beyond the first reactant to handle this conversion correctly.
>
> You can also provide a **Python callable** instead of an `ArrheniusRateConstant`:
> ```python
> rate_constant=lambda T: 7.45e7 * C_H2O_M * math.exp(-4430.0 * (1/T - 1/298.15))
> ```
> This is useful for rate expressions that don't follow a simple Arrhenius form.

### 5c. `DissolvedReversibleReaction` — Reversible Aqueous Reaction

Like `DissolvedReaction`, but with forward and reverse rates. You can specify either:
- `forward_rate_constant` + `reverse_rate_constant`, or
- `forward_rate_constant` + `equilibrium_constant`

In [8]:
# Example: SO2(aq) ⇌ HSO3- + H+  (reversible dissociation, kinetic form)
ka1_forward = DissolvedReversibleReaction(
    phase_name="AQUEOUS",
    reactant_names=["SO2_aq"],
    product_names=["HSO3m", "Hp"],
    solvent_name="H2O",
    forward_rate_constant=ArrheniusRateConstant(a=1.0e6, c=2090.0),
    equilibrium_constant=EquilibriumConstant(a=1.7e-2 / C_H2O_M, c=2090.0),
)
print("DissolvedReversibleReaction created.")

DissolvedReversibleReaction created.


> **When to use reversible reactions vs. constraints**: If the equilibrium timescale is much shorter than the solver timestep, use a `DissolvedEquilibriumConstraint` instead (see Section 6b). Reversible reactions are appropriate when the approach-to-equilibrium dynamics matter — for example, if the forward rate is slow enough that the system doesn't reach equilibrium within one timestep.

## 6. Constraints: Algebraic Equations for the DAE Solver

Constraints replace a species' ODE with an algebraic equation. The species whose ODE is replaced is called the **algebraic species** — its concentration is determined by the constraint equation rather than by time integration.

We'll build up a system incrementally, adding one constraint at a time. This is **the recommended development workflow** — it makes debugging much easier because each step can be validated in isolation.

### 6a. `HenryLawEquilibriumConstraint` — Instantaneous Gas-Liquid Partitioning

Enforces Henry's Law equilibrium between a gas-phase species and its dissolved form:

$$[A_{aq}] = K_H(T) \cdot R \cdot T \cdot f_v \cdot [A_g]$$

where $f_v = [H_2O] \cdot M_{w} / \rho$ is the liquid water volume fraction. The dissolved species (`condensed_species_name`) becomes the algebraic variable.

In [9]:
# Step 1: Henry's Law constraint only
hlc_so2 = HenryLawEquilibriumConstraint(
    gas_species_name="SO2",
    condensed_species_name="SO2_aq",
    solvent_name="H2O",
    condensed_phase_name="AQUEOUS",
    henrys_law_constant=HenrysLawConstant(
        hlc_ref=1.23 * M_ATM_TO_MOL_M3_PA,    # mol/(m³·Pa) at 298.15 K
        c=3120.0,                             # K (temperature dependence)
    ),
    mw_solvent=MW_H2O,     # kg/mol — needed for mol/m³_air ↔ mol/L_solution conversion
    rho_solvent=RHO_H2O,   # kg/m³
)

print(f"Henry's Law constraint for SO2")
print(f"  K_H(298 K) = {1.23 * M_ATM_TO_MOL_M3_PA:.4e} mol/(m³·Pa)")
print(f"  Temperature dependence: c = {3120.0} K")

Henry's Law constraint for SO2
  K_H(298 K) = 1.2139e-02 mol/(m³·Pa)
  Temperature dependence: c = 3120.0 K


> **Unit conversion note**: The `mw_solvent` and `rho_solvent` parameters are needed because MIAM state variables are in mol/m³ (of air for gas, of solution for aqueous), but the Henry's Law constant relates partial pressure to aqueous-phase molarity. These parameters allow MIAM to convert between the two conventions internally.

Let's test this single constraint in isolation to verify it works:

In [10]:
GAS0_SO2 = 3.01e-8  # mol/m³ (~1 ppb at cloud level)

# Compute expected partitioning
# MIAM's Henry's Law: [A_aq] = K_H * R * T * f_v * [A_g]
hlc_T = (1.23 * M_ATM_TO_MOL_M3_PA) * math.exp(3120.0 * (1.0/T_INIT - 1.0/T0))
alpha_SO2 = hlc_T * R_GAS * T_INIT * f_v
expected_g = GAS0_SO2 / (1 + alpha_SO2)
expected_aq = alpha_SO2 * expected_g

# Build minimal model: just HLC + mass conservation
model_step1 = Model(
    name="step1_hlc_only",
    species=all_species,
    condensed_phases=[aq_phase],
    representations=[cloud],
    processes=[],
    constraints=[
        hlc_so2,
        LinearConstraint(
            algebraic_phase_name="gas",
            algebraic_species_name="SO2",
            terms=[
                LinearConstraintTerm("gas", "SO2", 1.0),
                LinearConstraintTerm("AQUEOUS", "SO2_aq", 1.0),
            ],
            constant=GAS0_SO2,
        ),
    ],
)

micm = MICM(
    mechanism=mechanism,
    solver_type=SolverType.rosenbrock_dae4_standard_order,
    external_models=[model_step1],
)
state = micm.create_state()
model_step1.set_default_parameters(state)
state.set_conditions(temperatures=T_INIT, pressures=P_INIT)
state.set_concentrations({
    "SO2": expected_g,
    "CLOUD.AQUEOUS.SO2_aq": expected_aq,
    "CLOUD.AQUEOUS.H2O": C_H2O,
    "CLOUD.AQUEOUS.Hp": 1e-4,
    "CLOUD.AQUEOUS.OHm": 1e-8,
    "CLOUD.AQUEOUS.HSO3m": 1e-8,
})

# Solve for 1 second
result = micm.solve(state, time_step=1.0)
print(f"Solver state: {result.state.name}")

concs = state.get_concentrations()
so2_g_final = concs["SO2"][0]
so2_aq_final = concs["CLOUD.AQUEOUS.SO2_aq"][0]
ratio = so2_aq_final / so2_g_final

print(f"\nHenry's Law verification:")
print(f"  [SO2_aq] / [SO2_g] = {ratio:.6f}")
print(f"  Expected alpha     = {alpha_SO2:.6f}")
print(f"  Mass conservation: {so2_g_final + so2_aq_final:.4e} (expected {GAS0_SO2:.4e})")
print(f"  ✓ Henry's Law satisfied!" if abs(ratio/alpha_SO2 - 1) < 0.01 else "  ✗ Henry's Law NOT satisfied")

Solver state: Converged

Henry's Law verification:
  [SO2_aq] / [SO2_g] = 0.000017
  Expected alpha     = 0.000017
  Mass conservation: 3.0100e-08 (expected 3.0100e-08)
  ✓ Henry's Law satisfied!


### 6b. `DissolvedEquilibriumConstraint` — Aqueous Equilibrium

Enforces an equilibrium relationship between dissolved species:

$$K(T) = \frac{[\text{products}]}{[\text{reactants}] \cdot [\text{solvent}]}$$

One product species is chosen as the **algebraic species** — its ODE is replaced by this constraint. The choice of algebraic species affects numerical conditioning but not the final answer.

Common examples:
- **Water dissociation**: $\text{H}_2\text{O} \rightleftharpoons \text{H}^+ + \text{OH}^-$, with $K_w = 10^{-14}$ M²
- **SO₂ first dissociation**: $\text{SO}_2(aq) \rightleftharpoons \text{HSO}_3^- + \text{H}^+$

In [11]:
# Water dissociation: H2O ⇌ H+ + OH-
# Kw = 1e-14 M² in literature
# In MIAM units: Kw / C_H2O_M² (because [H2O] is in the denominator)
kw_constraint = DissolvedEquilibriumConstraint(
    phase_name="AQUEOUS",
    reactant_names=["H2O"],
    product_names=["Hp", "OHm"],
    algebraic_species_name="OHm",  # OH- concentration determined by constraint
    solvent_name="H2O",
    equilibrium_constant=EquilibriumConstant(
        a=1e-14 / (C_H2O_M * C_H2O_M),  # convert from M² to MIAM units
        c=0.0,  # no temperature dependence for simplicity
    ),
)

# SO2 first dissociation: SO2(aq) ⇌ HSO3- + H+
# Ka1 = 1.7e-2 M in literature
ka1_constraint = DissolvedEquilibriumConstraint(
    phase_name="AQUEOUS",
    reactant_names=["SO2_aq"],
    product_names=["HSO3m", "Hp"],
    algebraic_species_name="HSO3m",  # HSO3- determined by constraint
    solvent_name="H2O",
    equilibrium_constant=EquilibriumConstant(
        a=1.7e-2 / C_H2O_M,  # convert from M to MIAM units
        c=2090.0,             # K (temperature dependence)
    ),
)

print("Dissolved equilibrium constraints defined.")
print(f"  Kw: OHm is the algebraic species")
print(f"  Ka1: HSO3m is the algebraic species")

Dissolved equilibrium constraints defined.
  Kw: OHm is the algebraic species
  Ka1: HSO3m is the algebraic species


> **Choosing the algebraic species**: Any product species can be the algebraic variable. The usual guideline is to pick the species that appears in fewer other constraint equations, to reduce coupling in the Jacobian. For Kw, choosing OH⁻ is conventional since it typically doesn't appear in other equilibria.

### 6c. `LinearConstraint` — Mass Conservation and Charge Balance

Enforces a linear relationship between species concentrations:

$$\sum_i c_i \cdot [S_i] = \text{constant}$$

where $c_i$ are coefficients and $[S_i]$ are species concentrations. One species is algebraic — its value is determined from the constraint.

Two canonical uses:
1. **Mass conservation**: Total atoms of an element are constant
2. **Charge balance**: Total positive charge equals total negative charge

In [12]:
# Mass conservation for sulfur: SO2(g) + SO2(aq) + HSO3- = total S
mass_S = LinearConstraint(
    algebraic_phase_name="gas",
    algebraic_species_name="SO2",     # SO2(g) is determined by mass balance
    terms=[
        LinearConstraintTerm("gas", "SO2", 1.0),
        LinearConstraintTerm("AQUEOUS", "SO2_aq", 1.0),
        LinearConstraintTerm("AQUEOUS", "HSO3m", 1.0),
    ],
    constant=GAS0_SO2,  # total initial sulfur (mol/m³)
)

# Charge balance: H+ = OH- + HSO3-
# Written as: H+ - OH- - HSO3- = 0
charge = LinearConstraint(
    algebraic_phase_name="AQUEOUS",
    algebraic_species_name="Hp",       # H+ is determined by charge balance
    terms=[
        LinearConstraintTerm("AQUEOUS", "Hp", 1.0),
        LinearConstraintTerm("AQUEOUS", "OHm", -1.0),
        LinearConstraintTerm("AQUEOUS", "HSO3m", -1.0),
    ],
    constant=0.0,  # no background charge
)

print("Linear constraints defined.")
print(f"  Mass S: SO2(g) is the algebraic species, total = {GAS0_SO2:.2e} mol/m³")
print(f"  Charge: Hp is the algebraic species, net charge = 0")

Linear constraints defined.
  Mass S: SO2(g) is the algebraic species, total = 3.01e-08 mol/m³
  Charge: Hp is the algebraic species, net charge = 0


> **The `constant` field**: For mass conservation, this is the total budget. If you have background species that contribute to the budget but aren't in the constraint (e.g., SO₄²⁻ from a previous timestep), include their contribution in `constant`. For charge balance, `constant=0.0` enforces electroneutrality, but you could set it to a nonzero value to account for background ions.

## 7. Building Incrementally: The Recommended Workflow

Now let's put it all together, adding constraints one at a time and verifying each step. This is the single most important debugging strategy for MIAM configurations — if something goes wrong, it's in the last piece you added.

In [13]:
def compute_ics():
    """Compute self-consistent initial conditions for the 5-species system."""
    T = T_INIT
    hlc_T = (1.23 * M_ATM_TO_MOL_M3_PA) * math.exp(3120.0 * (1.0/T - 1.0/T0))
    alpha = hlc_T * R_GAS * T * f_v
    Ka1_T = (1.7e-2 / C_H2O_M) * math.exp(2090.0 * (1.0/T0 - 1.0/T))
    Kw_T = 1e-14 / (C_H2O_M * C_H2O_M)
    
    # Iterate on [H+]
    hp_ic = 1e-4  # initial guess
    for _ in range(100):
        ohm_ic = Kw_T * C_H2O * C_H2O / hp_ic
        f = 1.0 + alpha + Ka1_T * alpha * C_H2O / hp_ic
        so2_g_ic = GAS0_SO2 / f
        so2_aq_ic = alpha * so2_g_ic
        hso3m_ic = Ka1_T * so2_aq_ic * C_H2O / hp_ic
        hp_new = ohm_ic + hso3m_ic
        if abs(hp_new - hp_ic) < 1e-15 * hp_ic:
            break
        hp_ic = 0.5 * (hp_ic + hp_new)
    
    return {
        "SO2": so2_g_ic,
        "CLOUD.AQUEOUS.SO2_aq": so2_aq_ic,
        "CLOUD.AQUEOUS.Hp": hp_ic,
        "CLOUD.AQUEOUS.OHm": ohm_ic,
        "CLOUD.AQUEOUS.HSO3m": hso3m_ic,
        "CLOUD.AQUEOUS.H2O": C_H2O,
    }

ics = compute_ics()
print("Self-consistent initial conditions:")
for name, val in ics.items():
    print(f"  {name:30s} = {val:.4e} mol/m³")
print(f"\n  pH = {-math.log10(ics['CLOUD.AQUEOUS.Hp'] / 1000):.2f}")

Self-consistent initial conditions:
  SO2                            = 2.8851e-08 mol/m³
  CLOUD.AQUEOUS.SO2_aq           = 4.8198e-13 mol/m³
  CLOUD.AQUEOUS.Hp               = 1.2495e-09 mol/m³
  CLOUD.AQUEOUS.OHm              = 7.2025e-13 mol/m³
  CLOUD.AQUEOUS.HSO3m            = 1.2488e-09 mol/m³
  CLOUD.AQUEOUS.H2O              = 1.6667e-02 mol/m³

  pH = 11.90


In [14]:
# Full model: HLC + Kw + Ka1 + mass conservation + charge balance
model_full = Model(
    name="step_by_step",
    species=all_species,
    condensed_phases=[aq_phase],
    representations=[cloud],
    processes=[],  # No kinetics for now
    constraints=[hlc_so2, kw_constraint, ka1_constraint, mass_S, charge],
)

micm = MICM(
    mechanism=mechanism,
    solver_type=SolverType.rosenbrock_dae4_standard_order,
    external_models=[model_full],
)
state = micm.create_state()
model_full.set_default_parameters(state)
state.set_conditions(temperatures=T_INIT, pressures=P_INIT)
state.set_concentrations(ics)

# Solve for 10 seconds in a single step
result = micm.solve(state, time_step=10.0)
assert result.state == SolverState.Converged, "Solver failed"

# Validate all constraints
concs = state.get_concentrations()
so2_g = concs["SO2"][0]
so2_aq = concs["CLOUD.AQUEOUS.SO2_aq"][0]
hp_f = concs["CLOUD.AQUEOUS.Hp"][0]
ohm_f = concs["CLOUD.AQUEOUS.OHm"][0]
hso3m_f = concs["CLOUD.AQUEOUS.HSO3m"][0]
h2o_f = concs["CLOUD.AQUEOUS.H2O"][0]

hlc_T = (1.23 * M_ATM_TO_MOL_M3_PA) * math.exp(3120.0 * (1.0/T_INIT - 1.0/T0))
alpha_expected = hlc_T * R_GAS * T_INIT * f_v
Ka1_T = (1.7e-2 / C_H2O_M) * math.exp(2090.0 * (1.0/T0 - 1.0/T_INIT))
Kw_T = 1e-14 / (C_H2O_M * C_H2O_M)

print("Constraint validation after 10 s:")
print(f"  Henry's Law:  aq/g = {so2_aq/so2_g:.6f}  (expected {alpha_expected:.6f})")
print(f"  Ka1:          Q = {hso3m_f*hp_f/(so2_aq*h2o_f):.4e}  (expected {Ka1_T:.4e})")
print(f"  Kw:           Q = {hp_f*ohm_f/(h2o_f*h2o_f):.4e}  (expected {Kw_T:.4e})")
print(f"  Mass S:       total = {so2_g+so2_aq+hso3m_f:.4e}  (expected {GAS0_SO2:.4e})")
print(f"  Charge:       H+ - OH- - HSO3- = {hp_f-ohm_f-hso3m_f:.4e}  (expected 0)")
print(f"  pH = {-math.log10(hp_f / 1000):.2f}")
print("\n  All constraints satisfied! ✓")

Constraint validation after 10 s:
  Henry's Law:  aq/g = 0.000017  (expected 0.000017)
  Ka1:          Q = 1.9426e-04  (expected 1.9426e-04)
  Kw:           Q = 3.2399e-18  (expected 3.2399e-18)
  Mass S:       total = 3.0100e-08  (expected 3.0100e-08)
  Charge:       H+ - OH- - HSO3- = 0.0000e+00  (expected 0)
  pH = 11.90

  All constraints satisfied! ✓


## 8. Caveats and Debugging Tips

DAE systems are powerful but can be tricky. Here are the most common issues and how to handle them.

### 8a. Inconsistent Initial Conditions

Every algebraic constraint $G(y) = 0$ must be approximately satisfied at $t = 0$. If $G(y_0)$ is large, the solver's first Newton correction overshoots wildly.

**Good news**: MICM's DAE solver now automatically performs Newton iteration on the algebraic variables before time-stepping to project them onto the constraint manifold. This means a rough initial guess is usually sufficient.

**But watch out** for systems with very small equilibrium constants (like $K_w \approx 10^{-14}$) — the constraint initialization may need more iterations or tighter tolerance. If the solver fails, try:

1. Providing better initial guesses (compute them from the constraint equations)
2. Using the damped fixed-point iteration shown in `compute_ics()` above

### 8b. Unit Conversion Errors

This is the single most common source of bugs in MIAM configurations.

MIAM state variables are in **mol/m³** (of air for gas-phase, of solution for aqueous-phase), but literature equilibrium constants are usually in **mol/L**. The conversion factor is **1000** per concentration power.

For a dissociation $A \rightleftharpoons B + C$ with $K_{eq}$ in mol/L:
- The MIAM equilibrium constant needs to be divided by `C_H2O_M` (55.556 mol/L)
- Missing this introduces a factor of $10^{1.74}$ per concentration term
- For $K_w$ with two products, the error is $10^{3.5}$ — easily enough to crash the solver

**Always double-check your unit conversions against the literature values.**

### 8c. Negative Concentrations

Neither Rosenbrock nor BDF solvers enforce non-negativity. Large, fast reactions can drive a species below zero within a single solver stage.

**Symptoms**: A species that should asymptotically approach zero goes slightly negative, then subsequent steps amplify through rate expressions (e.g., $k[A][B]$ where $[B] < 0$ flips the sign).

**Fix**: Reduce the timestep in the critical region, or clamp concentrations to zero at the physics boundary.

### 8d. The Solver Reports "Converged" but the Answer is Wrong

The Rosenbrock error estimator can be fooled when constraint residuals are enormous (see the [debugging guide](https://github.com/NCAR/miam/blob/main/docs/src/user_guide/debugging_dae_systems.rst) for a detailed case study). **Always validate**:

1. **Mass conservation**: Sum the species that should be conserved and compare before vs. after
2. **Charge balance**: Compute $[H^+] - [OH^-] - [\text{anions}]$ — it should be near zero
3. **Equilibrium ratios**: Check that $Q = K$ for each equilibrium constraint
4. **Signs**: All concentrations must be non-negative

The solver giving `SolverState.Converged` is necessary but **not sufficient** — always do your own physical validation.

### 8e. Debugging Checklist

When something goes wrong:

1. **Build incrementally**: Start with one constraint. Add complexity one piece at a time.
2. **Compare formulations**: A kinetic ODE and equivalent DAE constraint should agree at steady state. Disagreement → constraint math is wrong.
3. **Check units**: mol/m³ vs mol/L is the most common error.
4. **Provide reasonable initial guesses**: Correct sign, correct order of magnitude.

## 9. Summary

| Component | Purpose | When to Use |
|---|---|---|
| `UniformSection` | Cloud droplet geometry | Simple cloud chemistry |
| `SingleMomentMode` | Log-normal aerosol mode | Prescribed aerosol sizes |
| `TwoMomentMode` | Prognostic aerosol mode | Evolving aerosol distributions |
| `HenryLawPhaseTransfer` | Kinetic gas-to-aqueous transfer | Slow transfer (large drops, low accommodation) |
| `DissolvedReaction` | Irreversible aqueous reaction | Oxidation, hydrolysis |
| `DissolvedReversibleReaction` | Reversible aqueous reaction | Approach-to-equilibrium dynamics |
| `HenryLawEquilibriumConstraint` | Instantaneous partitioning | Fast transfer (cloud drops) |
| `DissolvedEquilibriumConstraint` | Aqueous equilibrium | Dissociation, ionization |
| `LinearConstraint` | Mass conservation / charge balance | Always |

The golden rule: **build incrementally, validate each step, check your units.**

In the [next tutorial](14.%20cam_cloud_chemistry.ipynb), we'll put all of these pieces together into a full CAM Cloud Chemistry simulation with gas-phase precursor reactions, Henry's Law partitioning, dissociation equilibria, and sulfate production kinetics.